# QDGrasp Phase 3.4 — CUDA backend decision

This notebook resolves one question: **can MuJoCo Warp run the QDGrasp release hands?**

`P3.4-04` measured what the models require, on CPU:

| requirement | why it blocks the phase |
| --- | --- |
| `mjTRN_TENDON` | `shadow_hand` drives 4 of 20 actuators through tendons |
| `equality:mjEQ_WELD` | the `mocap-weld-v3` protocol drives the wrist through a welded mocap body |
| `mocap_body` | same protocol |
| per-contact force + frame | the safety budget is defined on resolved contact force |

If Warp cannot carry all four, Phase 3.4 **stays blocked** and a backend decision
record is written. Substituting a mock CUDA backend, or dropping Shadow from the
gate, is not an accepted resolution.

This notebook does **not** benchmark search throughput: the CUDA backend
(`P3.4-05`) does not exist yet, and reporting a CPU number as CUDA evidence is
exactly what the gate forbids.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

CODE_REVISION = "46de548334dd3808fc329b33c5e089c5b92b73d9"
MENAGERIE_REVISION = "da76818e269b82289eba39808e2fb91d679d6994"
REPO_URL = "https://github.com/ninicom/qdgrasp.git"
REPO_DIR = Path("/tmp/qdgrasp_repo")
ASSETS_DIR = Path("/tmp/robot-assets/mujoco-menagerie")

assert sys.version_info >= (3, 11), f"Python >=3.11 required, got {sys.version}"
os.environ.update(
    QDGRASP_ROBOT_ASSETS_ROOT="/tmp/robot-assets",
    MUJOCO_GL="egl",
    PYTHONHASHSEED="0",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
)

subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
    "lightning==2.6.5", "mujoco==3.12.0", "numpy==2.4.6", "scipy==1.17.1",
    "trimesh==4.12.2", "safetensors==0.8.0", "pydantic==2.13.4", "PyYAML==6.0.3",
    "einops==0.8.2", "rich==14.3.4", "typer==0.27.1", "torchmetrics==1.9.0",
    "Pillow==12.1.1", "pytest==9.1.1",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "--force-reinstall",
    f"git+{REPO_URL}@{CODE_REVISION}",
], check=True)

for directory, url, revision in (
    (REPO_DIR, REPO_URL, CODE_REVISION),
    (ASSETS_DIR, "https://github.com/google-deepmind/mujoco_menagerie.git", MENAGERIE_REVISION),
):
    if not directory.exists():
        directory.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", url, str(directory)], check=True)
    subprocess.run(["git", "-C", str(directory), "fetch", "--depth", "1", "origin", revision], check=True)
    subprocess.run(["git", "-C", str(directory), "checkout", "--detach", revision], check=True)
    actual = subprocess.check_output(["git", "-C", str(directory), "rev-parse", "HEAD"], text=True).strip()
    assert actual == revision, (directory, actual, revision)

print("Pinned QDGrasp revision:", CODE_REVISION)
print("Pinned Menagerie revision:", MENAGERIE_REVISION)


## 1. Refuse a CPU host

A CPU fallback is never admissible as CUDA evidence
(`docs/decisions/0006-cuda-hardware-required.md`). This cell fails the run rather
than continuing on CPU.


In [ ]:
import torch

assert torch.cuda.is_available(), "no CUDA device: this notebook must run on a GPU kernel"
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("capability:", f"{props.major}.{props.minor}")
print("VRAM GiB:", round(props.total_memory / (1024 ** 3), 2))
print("torch:", torch.__version__, "| cuda build:", torch.version.cuda)


## 2. Previous CUDA gates still pass

Plan section 10 requires the Phase 1 CUDA smoke and the Phase 2 FK parity to be
re-run before any Phase 3.4 measurement, so a regression in the foundation is
never reported as a Phase 3.4 result.


In [ ]:
import subprocess
import sys

for script, out in (
    ("scripts/phase1_cuda_smoke.py", "/tmp/phase1_cuda_evidence.json"),
    ("scripts/phase2_cuda_fk_parity.py", "/tmp/phase2_cuda_evidence.json"),
):
    print("=" * 70)
    print("running", script)
    completed = subprocess.run(
        [sys.executable, script, "--out", out],
        cwd="/tmp/qdgrasp_repo", capture_output=True, text=True,
    )
    print(completed.stdout[-3000:])
    print(completed.stderr[-2000:], file=sys.stderr)
    assert completed.returncode == 0, f"{script} failed with {completed.returncode}"


## 3. Install MuJoCo Warp

Not pinned in the repository locks yet: this notebook is the spike that decides
whether it earns a pin. The install is reported, not assumed to succeed.


In [ ]:
import subprocess
import sys

warp_install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "warp-lang", "mujoco-warp"],
    capture_output=True, text=True,
)
print("install returncode:", warp_install.returncode)
print(warp_install.stdout[-2000:])
print(warp_install.stderr[-2000:], file=sys.stderr)

import importlib.util
for module in ("warp", "mujoco_warp", "mujoco.mjx"):
    print(f"{module:14s}", "available" if importlib.util.find_spec(module) else "NOT installed")


## 4. Requirement matrix and backend verdict

The spike reports what the models need; the gate script decides. A verdict other
than `supported` exits nonzero and keeps Phase 3.4 blocked.


In [ ]:
import json
import subprocess
import sys

spike = subprocess.run(
    [sys.executable, "scripts/phase3_4_backend_spike.py", "--out", "/tmp/phase3_4_requirements.json"],
    cwd="/tmp/qdgrasp_repo", capture_output=True, text=True,
)
print(spike.stdout[-4000:])
assert spike.returncode == 0, spike.stderr[-2000:]

gate = subprocess.run(
    [sys.executable, "scripts/phase3_4_cuda_contact_search.py",
     "--device", "cuda:0", "--profile", "kaggle-t4-micro",
     "--evidence", "/tmp/phase3_4_cuda_evidence.json"],
    cwd="/tmp/qdgrasp_repo", capture_output=True, text=True,
)
print(gate.stdout[-6000:])
print(gate.stderr[-3000:], file=sys.stderr)

evidence = json.loads(open("/tmp/phase3_4_cuda_evidence.json", encoding="utf-8").read())
resolution = evidence["backend_resolution"]
print()
print("VERDICT:", resolution["verdict"])
print("unsupported:", resolution.get("unsupported", []))
print("gate exit:", gate.returncode)


## 5. Compute Sanitizer on a minimized reproducer

Section 3.3 step 5. The classification already excluded a benchmark reading
defect and an index-range error, and showed the rejected set changes between
runs, which points at a race or uninitialized memory. Sanitizer output is what
separates those two; contact numbers cannot.

Small on purpose: sanitizer costs one to two orders of magnitude, so a
1024-world run would not finish. This is a diagnostic and never performance
evidence.


In [ ]:
import shutil
import subprocess
import sys

sanitizer = shutil.which("compute-sanitizer")
print("compute-sanitizer:", sanitizer or "NOT FOUND")

plain = subprocess.run(
    [sys.executable, "scripts/phase3_4_1_sanitizer.py", "--worlds", "8",
     "--horizon", "20", "--out", "/tmp/repro_plain.json"],
    cwd="/tmp/qdgrasp_repo", capture_output=True, text=True,
)
print("=== reproducer without sanitizer ===")
print(plain.stdout[-1500:])

if not sanitizer:
    print("compute-sanitizer unavailable; the question stays open rather than guessed at.")
else:
    for tool, extra in (("racecheck", []), ("initcheck", ["--print-limit", "40"])):
        print("=" * 70)
        print("compute-sanitizer --tool", tool)
        run = subprocess.run(
            [sanitizer, "--tool", tool, "--error-exitcode", "0", *extra,
             sys.executable, "scripts/phase3_4_1_sanitizer.py",
             "--worlds", "4", "--horizon", "8"],
            cwd="/tmp/qdgrasp_repo", capture_output=True, text=True, timeout=5400,
        )
        out = run.stdout + run.stderr
        records = [ln for ln in out.splitlines() if ln.lstrip().startswith("=========")]
        print(f"--- {len(records)} sanitizer report lines; first 60 verbatim ---")
        for ln in records[:60]:
            print(ln.strip()[:200])
        print(f"--- last 6 ---")
        for ln in records[-6:]:
            print(ln.strip()[:200])


In [ ]:
import shutil
import subprocess
import sys

# Section 3.3 step 6: does the defect reproduce in upstream MuJoCo Warp alone?
# If it does, this is not a QDGrasp wrapper problem and the fix is a pinned
# patch or version. If only QDGrasp reproduces it, the wrapper is at fault.
print("=" * 70)
print("upstream mjwarp-testspeed on the same model")

testspeed = shutil.which("mjwarp-testspeed")
print("mjwarp-testspeed:", testspeed or "not on PATH; trying module form")

import mujoco_warp
print("mujoco_warp:", getattr(mujoco_warp, "__version__", "unknown"))

# Export the exact release model so upstream sees what QDGrasp sees.
sys.path.insert(0, "/tmp/qdgrasp_repo")
from qdgrasp.dataset.pipeline.generated_reachable import build_generated_reachable_object
from qdgrasp.dataset.pipeline.validators.mujoco_rollout import build_rollout_scene_model
from qdgrasp.robot.spec import RobotSpec, resolve_robot_asset

spec = RobotSpec.from_config("leap_hand.yaml", sample_anchors=False)
fixture = build_generated_reachable_object("leap_hand")
model = build_rollout_scene_model(
    resolve_robot_asset(spec.config.source_asset),
    fixture.collision_geoms,
    object_pos=fixture.object_pos,
    object_mass=fixture.mass,
)
print("model: ngeom", model.ngeom, "nq", model.nq, "nu", model.nu)

sanitizer = shutil.which("compute-sanitizer")
cmd = [sys.executable, "-c", (
    "import mujoco, mujoco_warp, numpy as np, sys;"
    "sys.path.insert(0,'/tmp/qdgrasp_repo');"
    "from qdgrasp.dataset.pipeline.generated_reachable import build_generated_reachable_object as f;"
    "from qdgrasp.dataset.pipeline.validators.mujoco_rollout import build_rollout_scene_model as b;"
    "from qdgrasp.robot.spec import RobotSpec, resolve_robot_asset;"
    "s=RobotSpec.from_config('leap_hand.yaml', sample_anchors=False);"
    "x=f('leap_hand');"
    "m=b(resolve_robot_asset(s.config.source_asset), x.collision_geoms,"
    " object_pos=x.object_pos, object_mass=x.mass);"
    "d=mujoco.MjData(m); mujoco.mj_forward(m,d);"
    "wm=mujoco_warp.put_model(m); wd=mujoco_warp.put_data(m,d,nworld=4);"
    "[mujoco_warp.step(wm,wd) for _ in range(8)];"
    "print('upstream stepped 8 times, 4 worlds')"
)]

if sanitizer:
    run = subprocess.run(
        [sanitizer, "--tool", "initcheck", "--error-exitcode", "0",
         "--print-limit", "6", *cmd],
        capture_output=True, text=True, timeout=5400,
    )
    out = run.stdout + run.stderr
    recs = [ln.strip() for ln in out.splitlines() if ln.lstrip().startswith("=========")]
    print(f"--- upstream-only path: {len(recs)} sanitizer lines ---")
    for ln in recs[:40]:
        print(ln[:190])
else:
    print("compute-sanitizer unavailable")


## 6. What this run does and does not establish

A `supported` verdict unblocks `P3.4-05` (the CUDA backend). It does **not**
close Phase 3.4: throughput, VRAM, CPU/GPU parity fixtures, a CPU-confirmed
finalist per hand and the ContactRich dataset are all still outstanding.

A blocked verdict is a legitimate result. Record it and write the backend
decision record; do not work around it.


In [ ]:
import json

evidence = json.loads(open("/tmp/phase3_4_cuda_evidence.json", encoding="utf-8").read())
print(json.dumps(evidence, indent=2, sort_keys=True))
